In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

# 提前把千问大模型和 Embedding 雷达加载到内存中待命
llm = OllamaLLM(model="qwen2.5:0.5b") 
embedding_radar = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

print("✅ 后勤装配完毕！雷达与大模型已在内存中全天候待命！")

In [ ]:
# 1. 准备机密文本 (可以随时在这里修改文本内容)
secret_text = """
北京时间 2026 年 3 月 17 日凌晨两点半，当英伟达 CEO 黄仁勋穿着那件标志性的黑色皮衣踏上 SAP 中心的舞台时，台下近万名开发者心里清楚：这一次，老黄要讲的不是某个单一芯片，而是一整套 AI“全家桶”。
这场演讲之前，市场早已躁动不安。黄仁勋曾在 2 月预告将发布一款“前所未见的芯片”，被普遍认为是采用台积电 1.6nm 制程、引入光通信技术的下一代 Feynman 架构。
今天揭晓的 Feynman 架构、Vera Rubin 平台的量产进展，以及面向企业级自主代理的开源平台 NemoClaw，不过是这座基础设施落地所需的“钢筋水泥”。黄仁勋用两个多小时向资本市场证明了一件事：英伟达早已不是那个卖显卡的公司，现在的英伟达是一家为“数万亿美元 AI 基建时代”搭建完整技术栈的“总包工头”。
1
 回顾 CUDA 20 年：安装基数引爆飞轮，GPU 算力成本持续下降
演讲刚开始，黄仁勋用近十分钟篇幅，回溯了 CUDA 架构诞生 20 年的演进历程。他将这套软件生态定义为英伟达一切业务的“中心”，并直言：真正难以复制的壁垒，是底层的安装基数。
“二十年来，我们一直致力于这种革命性架构——单指令多线程（SIMT），让开发者编写的扩展代码能够轻松生成多线程应用，编程难度远低于传统方法。”黄仁勋回忆道。他特别提到近期引入的“tiles”（张量核心编程块）功能，旨在帮助开发者调用 Tensor Core 以及支撑当今 AI 的数学结构。
如今，围绕 CUDA 已形成庞大的工具链生态：数千种工具、编译器、框架和库；仅开源领域就有数十万个公开项目。“CUDA 真正融入了每一个生态系统，”黄仁勋说，“这张图，基本上描述了 100% 的媒体战略——你们从一开始就看我讲这张幻灯片。”
图片
他随后指向图表底层：“最难实现的东西在这里——安装基数。我们花了 20 年，才在全球建立起数亿块运行 CUDA 的 GPU 和计算系统。我们在每一朵云里，在每一家计算机公司里，服务几乎每一个行业。”
黄仁勋将这套逻辑总结为“飞轮效应”：
“CUDA 的安装基数，是飞轮加速的原因。安装基数吸引开发者，开发者创造新算法，带来技术突破——比如深度学习，还有很多其他领域。这些突破催生全新市场，围绕它们形成新的生态，更多公司加入，进而扩大安装基数。”
他称这一飞轮正在加速：“NVIDIA 库的下载量增长极快，规模巨大，增速前所未有。这个飞轮让计算平台能够承载如此多的应用和突破，更重要的是，它赋予了这些基础设施极长的有效生命周期。”
背后的逻辑很直接：CUDA 支持的应用程序范围足够广，覆盖 AI 生命周期的每一个阶段，对接每一种数据处理平台，加速各类科学原理求解器。“一旦安装 NVIDIA GPU，它的使用寿命极高。这也是为什么六年前出货的 Ampere 架构，在云上的定价反而在上涨。”
当安装基数足够大、飞轮足够快、开发者触达足够广，并且软件持续更新时，结果就是计算成本不断下降。
“加速计算让应用性能大幅提升，而我们在其生命周期内持续优化软件，”黄仁勋总结道，“你不仅获得初期的性能跃升，还能获得持续的算力成本降低。我们愿意支持全球每一块 GPU，因为它们架构兼容。为什么愿意？因为安装基数足够大——每发布一项新优化，数百万用户受益。”
他最后补充道：“这套动态机制，让 NVIDIA 架构不断扩展应用范围，加速自身增长，同时降低计算成本，最终催生新的增长。CUDA 就在这一切的中心。”
作为结尾的轻松注脚，黄仁勋提及了 GeForce 的历史：“我知道你们中有多少人是从 GFORCE 成长起来的——那是最棒的市场营销，我们在很早之前就开始吸引未来的客户。”
2
 为数据处理打造新的核心软件库
演讲中，老黄提到下面这张图是本场演讲中最重要的一张图，因为里面提到了英伟达为数据处理打造的新的核心软件库。老黄在演讲中谈到，随着 AI 的快速发展，全球数据处理体系正迎来一次结构性的变革，其中最核心的变化，是 结构化数据与非结构化数据的全面加速。
图片
黄仁勋指出，长期以来企业计算的基础建立在 结构化数据 之上。无论是 SQL、Spark、Pandas 等技术体系，还是诸如 Snowflake、Databricks、Amazon 的 EMR、Microsoft 的 Azure Fabric，以及 Google 的 BigQuery 等大型数据平台，本质上都在处理一种核心数据结构——数据框（DataFrame）。这些数据框可以被理解为巨型电子表格，承载着企业运营和业务决策所依赖的关键信息，是企业计算体系中的“事实来源”（ground truth）。
过去，对结构化数据的加速主要是为了提升企业的数据分析效率：让计算任务完成得更多、成本更低，并且能够在一天内更频繁地运行数据处理流程，从而让企业运营更加高效、更加同步。但在 AI 时代，这一逻辑正在发生变化。黄仁勋表示，未来不仅人类会使用这些数据结构，AI 系统和智能体（Agent）也将直接访问和使用结构化数据库，而 AI 的处理速度远远快于人类，这意味着数据处理基础设施必须获得数量级的性能提升。
图片
与此同时，另一类更庞大的数据也正在成为 AI 时代的重要资源——非结构化数据。黄仁勋指出，向量数据库、PDF 文档、视频、语音和演讲内容等都属于非结构化数据。全球每年产生的数据中，大约 90% 都是非结构化数据。然而在很长一段时间里，这些数据几乎无法被计算系统有效利用，人们只是阅读这些内容，然后把它们存储在文件系统中，却很难对其进行查询和搜索。其根本原因在于，非结构化数据缺乏可直接建立索引的结构，要使用这些数据，首先必须理解其 语义和目的。
而 AI 的多模态理解能力正在改变这一状况。正如 AI 已经在多模态感知和理解方面取得突破一样，同样的技术可以用于读取 PDF、理解视频和语音内容，并将其语义信息嵌入到可计算的数据结构中，从而使这些数据能够被搜索、查询和分析。换句话说，AI 正在把原本难以利用的海量非结构化数据转化为可计算的信息资源。
为了支持这一转变，NVIDIA 构建了两项关键基础技术。黄仁勋表示，就像当年为 3D 图形计算推出 RTX 技术一样，NVIDIA 现在为数据处理打造了新的核心软件库。其中 cuDF 用于加速数据框计算，主要面向结构化数据处理；而 cuVS 则面向向量存储和语义数据，用于处理非结构化数据和 AI 数据。这两项技术将成为未来数据基础设施中最重要的平台之一。
黄仁勋透露，目前这两项技术正在逐步融入全球复杂的数据处理生态系统。由于数据处理产业已经发展了数十年，围绕它已经形成了大量公司、平台和服务，因此将新的加速技术深度整合进整个生态需要时间。但 NVIDIA 已经看到越来越多的合作伙伴开始采用这些技术。
例如，IBM——SQL 的发明者之一，也是历史上最重要数据库技术的推动者——正在利用 cuDF 来加速其数据平台 IBM watsonx.data。在黄仁勋看来，这类合作标志着 AI 正在逐步重塑整个数据处理基础设施，使企业能够同时高效利用结构化数据和海量的非结构化数据。
3
 AI 原生行业的爆发和英伟达万亿美金的信心
AI 重塑整个基础设施的另一个标志是涌现出海量的 AI 原生企业。去年，这个行业经历了史无前例的飞跃。风险投资对初创公司的投入高达 1500 亿美元，创人类历史之最。投资规模也从千万美元级跃升至数十亿美金级。究其原因，是这些公司历史性地都需要海量算力和 Token。无论它们是创造 Token 还是为 Token 增值，它们对算力的渴望是共同的。
黄仁勋表示，“正如 PC、互联网、移动云革命催生了谷歌、亚马逊和 Meta，这次计算平台的迁移也将孕育出一批对世界未来至关重要的新巨头。”
那为什么这种 AI 企业的爆发会发生在这两年？黄仁勋称这因为发生了三件大事：
ChatGPT 开启生成式 AI 时代：
 计算从“基于检索”转向“基于生成”。这彻底改变了计算机的架构、供应和建设方式。
推理 AI（o1/o3）的出现：
 AI 开始拥有反思、规划、拆解问题的能力。o1 让生成式 AI 变得可靠且基于事实。为了“思考”，输入和输出 Token 的使用量呈爆炸式增长。
Claude Code 开启代理（Agentic）时代：
 这是首个代理模型。它能阅读文件、编码、编译、测试并迭代。它革新了软件工程。现在 NVIDIA 内部每个工程师都在使用 AI 代理辅助编程。
图片
他表示，“AI 已经从“感知”进化到“生成”，再到“推理”，现在已经可以执行极其高效的实际工作。 “推理拐点”已经到来。 AI 要思考、要行动、要阅读、要推理，每一环都在进行推理（Inference）。现在已经远超训练阶段，进入了推理的疆场。过去两年，计算需求增长了约 10,000 倍，而使用量增长了约 100 倍。我深感这两年的计算需求实际增长了 100 万倍。”
紧接着，老黄又分享了几个数据，让现场的气氛达到了高潮。
他高兴地向观众分享道，“去年我说 Blackwell 和 Rubin 到 2026 年的订单额将达 5000 亿美元，你们可能没觉得惊艳。但今天，在这里，我预见通过 2027 年的营收将至少达到 1 万亿美元。”
他进一步强调了这不是空话，因为计算需求只会更高。2025 年是 NVIDIA 的“推理之年”，NVIDIA 系统是全球成本最低的 AI 基础设施——使用寿命越长，成本就越低。
他这么说的背后有着坚实的数据支撑。目前，全球三分之一的 AI 计算开源模型（如 Anthropic 和 Meta 的模型）都跑在英伟达芯片上。NVIDIA 是全球唯一能运行 AI 所有领域的平台：语言、生物、图形、视觉、机器人、边缘或云端。
在英伟达的业务中，60% 来自顶级云服务商（Hyperscalers），不仅支持其内部 AI 消费（如推荐系统、搜索向大模型的迁移），更通过英伟达的生态系统加速每一家 AI 实验室。另外 40% 则遍布区域云、主权云、企业级服务器及工业自动化。
图片
4
 Token 成本全球最低
黄仁勋介绍了 NVIDIA 在 AI 推理基础设施上的最新进展。他表示，AI 性能的突破并不仅来自单一技术，而是由计算架构、软件栈和算法的系统级协同设计共同推动。
黄仁勋提到，NVIDIA 推出的 NVFP4（FP4）计算体系不仅是一种更低精度的数据格式，而是一种全新的 Tensor Core 计算架构。通过 NVFP4，NVIDIA 已经实现了在推理阶段几乎不损失精度的情况下，大幅提升性能和能效，同时这一计算格式也开始应用于模型训练。
结合 NVLink 72 高速互连，以及 Dynamo、TensorRT-LLM 等软件优化，NVIDIA 构建起一套面向大模型推理的完整技术体系。为优化底层软件与 GPU 内核，公司还投入数十亿美元建设 NVIDIA DGX Cloud 超级计算平台，用于开发和调优 AI 推理软件栈。
图片
黄仁勋强调，很多人曾认为推理是 AI 系统中最简单的部分，但实际上 推理既是最困难的环节，也是最关键的商业环节，因为它直接决定 AI 服务的收入来源。根据研究机构 SemiAnalysis 的评测，在数据中心层面，衡量 AI 系统效率的关键指标是每瓦特能够生成多少 token（tokens per watt）。由于数据中心受到电力等物理条件限制，本质上更像一个“AI 工厂”，企业必须在固定功率下尽可能多地生产 token。
评测结果显示，NVIDIA 在 AI 推理性能和效率上依然保持领先。按照传统 Moore's Law，新一代芯片通常只能带来约 1.5 倍性能提升，但从 Hopper H200 到 Grace Blackwell NVLink 72 架构，NVIDIA 的 每瓦特性能提升约 35 倍。SemiAnalysis 分析师 Dylan Patel 甚至认为实际提升接近 50 倍。这一架构也带来了更低的 token 成本，在当前市场上具有明显优势。
图片
黄仁勋表示，这种极致的软硬件协同设计还能显著提升现有系统性能。例如在部分 AI 推理平台中，仅通过更新 NVIDIA 软件栈，就能将生成速度从 约 700 token/ 秒提升至接近 5000 token/ 秒，性能提升约 7 倍。
黄仁勋强调，NVIDIA 的 Token 成本在世界范围内是“不可触碰”的。 即便竞争对手的架构是免费的，它也不够便宜。因为建立一个 1GW 的工厂，即便里面什么都不放，15 年的摊销成本也高达 400 亿美元。你必须确保在这个工厂里运行最强的计算机系统，才能获得最低的 Token 生产成本。
图片
在他看来，数据中心的角色正在发生变化：过去它是存储和计算中心，而未来将成为生产 token 的 AI 工厂。随着 AI 的普及，无论是云厂商、AI 公司还是传统企业，都将开始从“Token 工厂效率”的角度来衡量自己的计算基础设施，因为在 AI 时代，token 将成为新的数字商品，而计算能力则直接决定企业的价值创造能力。
5
 Vera Rubin 时代降临
整场演讲的另一个小高潮来自于 Vera Rubin 超级 AI 平台的亮相。
据介绍，这是一个全新的计算平台，由七款芯片组成， 涵盖计算、网络和存储三大功能，是目前最先进的 POD 规模 AI 平台。该平台包含 40 个机架、1.2 千万亿个晶体管、近 2 万个 NVIDIA 芯片、1152 个 NVIDIA Rubin GPU、60 exaflops 的运算能力以及 10 PB/s 的总扩展带宽。 该平台目前已全面投产，并得到了包括 Anthropic、OpenAI、Meta 和 Mistral AI 以及所有主要云提供商在内的众多客户的鼎力支持。
图片
他表示，过去十年间 AI 计算能力已经实现了 约 4000 万倍的提升，而这一变化正推动数据中心向“AI 超级计算机”形态演进。
“过去我发布产品时，可能只是手里举着一块芯片（比如 Hopper）；但现在，当我谈到 Vera Rubin 时，我说的是一个 全栈垂直整合的庞大系统。”
黄仁勋展示了 NVIDIA 最新的 Vera Rubin AI Supercomputer 系统，并强调这是一套从硬件到软件 完全纵向整合（vertically integrated） 的计算平台，专门为 Agentic AI（智能体 AI） 设计。随着大语言模型不断扩大规模、生成更多 token 并处理更长上下文，系统不仅需要更强的计算能力，还需要更高带宽的内存和存储访问能力，例如 KV Cache、结构化数据处理（cuDF）以及非结构化向量数据（cuVS）等。因此，NVIDIA 对整个系统架构进行了重新设计，包括计算、存储和网络。
图片
在硬件层面，NVIDIA 为这一平台开发了一款全新的数据中心 CPU——NVIDIA Vera CPU。该处理器针对极高的单线程性能、大规模数据处理能力以及能效进行了优化，并成为全球首个在数据中心中采用 LPDDR5 内存 的 CPU，从而实现领先的性能功耗比。黄仁勋透露，这款 CPU 已经开始单独销售，并有望成为 NVIDIA 的一项数十亿美元级业务。
在系统设计方面，Vera Rubin 超级计算机采用 100% 液冷架构，并通过 45°C 热水散热，大幅降低数据中心制冷成本。同时，系统内部布线被大幅简化，使得整机安装时间从过去的两天缩短至约两小时，从而显著提升数据中心部署效率。
网络互连是这一系统的核心技术之一。NVIDIA 在该平台上部署了第六代 NVLink 互连架构，实现 GPU 之间的高速扩展连接。黄仁勋表示，这是目前全球最先进、实现难度最高的大规模 GPU 互连系统之一。
图片
此外，NVIDIA 还推出了全球首个 CPO（Co-Packaged Optics）光电共封装 的 NVIDIA Spectrum-X Ethernet Switch，将光模块直接集成到芯片封装中，实现电子信号与光信号的直接转换，从而显著提升数据中心网络带宽与能效。这项技术由 NVIDIA 与台积电共同开发，目前已经进入量产阶段。
在更大规模的系统扩展上，NVIDIA 还展示了 Rubin Ultra Compute System。该系统通过新的 Kyber 机架架构，可以在一个 NVLink 域中连接 144 个 GPU，形成一台规模极大的统一计算机：前部为计算节点，后部为 NVLink 交换系统，通过中板结构连接，从而突破传统铜缆互连的距离限制。
图片
黄仁勋表示，随着 AI 模型规模和推理需求持续增长，未来的数据中心将越来越像一台 完整的超级计算机。而像 Vera Rubin 这样的系统，正是为下一代 AI 工作负载——尤其是智能体系统——而设计的核心计算基础设施。
图片
6
 下一代 AI 平台：Feynman 架构前瞻
此外，值得注意的是，NVIDIA 的 Feynman GPU 架构早在 2025 年 GTC 大会上就已得到确认。在本次演讲中，NVIDIA 列出了 Feynman GPU 与下一代 HBM、Vera CPU 以及构成 AI 数据中心基础的其他几个连接芯片。
图片
存储性能是制约 AI 推理的瓶颈，为此 NVIDIA 改变了以往使用标准 HBM 的策略，转而为 Feynman GPU 配备 定制化 HBM 技术。
超越标准：
现有的 Rubin 系列分别采用 HBM4 和 HBM4E，而 Feynman 将跳过通用规格，可能采用基于 HBM4E 的定制增强版 甚至提前布局 定制化 HBM5 方案。
深度整合：
这种定制化方案允许 NVIDIA 将部分 GPU 的数据处理逻辑直接嵌入存储底层的 Base Die（基础晶圆）中，从而实现超高的带宽与极低的延迟。
Feynman 平台将不再沿用目前的 Vera CPU 架构，而是确认搭载代号为 Rosa 的全新 CPU。
这种命名延续了 NVIDIA 以卓越女性科学家命名的传统，Rosa 架构致敬了美国物理学家、诺贝尔奖得主 罗莎琳·萨斯曼（Rosalyn Sussman Yalow），同时也呼应了发现 DNA 结构的 罗莎琳·富兰克林（Rosalind Franklin）。
Rosa CPU 被设计为 AI 智能体（Agentic AI）的编排中枢，旨在更高效地调度 GPU、存储与网络之间的 Token 流动，优化处理极其复杂的逻辑决策任务。
Feynman 时代标志着 NVIDIA 将 计算、存储和封装 三者进行了深度耦合。通过“3D 堆叠核心 + 定制化内存 + 专用 Rosa CPU”的组合，NVIDIA 正在将数据中心从传统的服务器集群演进为一台高度集成的“巨型超级计算机”。
图片
7
 推出 NVIDIA DSX——面向“AI 工厂”的基础设施平台
随后，黄仁勋还介绍了 AI 基础设施与数字孪生技术的发展，以及 NVIDIA 在其中的角色。
黄仁勋表示，当前 AI 基础设施的建设已经开始依赖完整的数字仿真体系。在数据中心建设阶段，系统会通过多种行业领先的工程仿真工具进行验证，例如使用 Siemens Simcenter STAR-CCM+ 进行外部热力学仿真、Cadence Design Systems 的相关工具进行内部热设计、ETAP 进行电力系统分析，以及 NVIDIA 自身的网络模拟平台 NVIDIA DSX Air。通过这些工具，可以在实际建设前完成“虚拟调试”（virtual commissioning），从而大幅缩短数据中心建设周期。
当数据中心正式投入运行后，其数字孪生系统会成为基础设施的“操作系统”。AI 智能体会与 NVIDIA DSX MaxQ 协同工作，动态调度整个基础设施：实时监控冷却、电力和网络系统，并不断优化计算吞吐量和能源效率。同时，AI 还可以根据电网实时负载和压力信号动态调整功率分配，从而在保证稳定性的同时提升整体效率。
黄仁勋表示，NVIDIA 及其合作伙伴正在全球范围内加速建设 AI 基础设施，以实现更高水平的可靠性、效率和计算吞吐能力。这一体系的核心平台就是 NVIDIA 新推出的 NVIDIA DSX——一个面向“AI 工厂”的基础设施平台。
图片
在数字孪生方面，NVIDIA 的 NVIDIA Omniverse 平台被设计用于承载全球规模的数字孪生模型。从地球级别的系统到各种规模的工业设施，未来都可以在这一平台上构建和运行数字孪生。黄仁勋特别感谢生态合作伙伴，并表示这些企业在过去几年中迅速加入 NVIDIA 的生态，共同建设可能是“世界上最大的计算系统”，并在全球范围内部署。
他还透露，NVIDIA 的 AI 计算基础设施正在向太空延伸。公司此前已经在卫星领域部署计算系统，并计划与合作伙伴开发新的太空计算平台 Vera Rubin Space One，用于在轨道上建设数据中心。由于太空环境中不存在对流或传导散热，只能通过辐射散热，因此系统冷却将成为一项极具挑战的工程问题，目前 NVIDIA 正在与工程团队共同研究解决方案。
图片
8
 联合 OpenClaw 之父推出 NemoClaw
整场演讲中对软件开发者影响最深远的部分是老黄对于最近爆火的“龙虾”现象的评论。黄仁勋高度评价了由 Peter Steinberger 创建的开源项目 OpenClaw。
黄仁勋表示，OpenClaw 的增长速度甚至超过了 Linux 在过去几十年的传播速度，其影响力“极其深远”。NVIDIA 也宣布将正式支持这一项目。
图片
黄仁勋提到，AI 大佬 Andrej Karpathy 最近提出的一种“AI 研究助手”模式很好地体现了智能体系统的能力：用户只需给 AI 一个任务，然后去休息，AI 便可以在后台自动运行数十甚至上百个实验，不断保留有效结果、淘汰无效方案。类似的案例正在不断出现。例如有人将 OpenClaw 安装在自己父亲的设备上，通过蓝牙连接酿酒设备，实现从生产流程到网站订单系统的全流程自动化，甚至在深圳已经出现用户排队购买相关产品的案例。随着这一项目迅速流行，社区甚至已经开始举办专门的开发者活动，足以说明其热度。
黄仁勋认为，从技术本质上看，OpenClaw 可以被理解为一种 智能体计算机的操作系统。它能够连接大语言模型，管理各种计算资源，并调用文件系统、工具和模型服务；同时具备任务调度能力，可以将复杂问题分解为多个步骤，并调用子智能体协同完成任务。此外，它还提供多模态输入输出能力，用户既可以通过文本、语音甚至手势与其交互，也可以通过消息、邮件等方式获得反馈。
正因如此，OpenClaw 的意义类似于过去的关键基础软件。黄仁勋表示，就像 Linux 让个人计算机和服务器生态得以发展，Kubernetes 推动了云计算时代的基础设施，而 HTML 构建了互联网应用基础一样，OpenClaw 为智能体时代提供了关键的软件栈。他认为，未来所有科技公司和软件公司都会面临一个问题——“你的 OpenClaw 战略是什么？” 因为企业软件正在从传统工具型软件，转向以智能体为核心的系统。
在传统企业 IT 架构中，数据中心主要负责存储数据和运行应用程序，各类软件系统通过工具和工作流为人类员工提供服务。但在智能体时代，这一模式将发生变化。黄仁勋认为，未来几乎所有 SaaS（Software as a Service） 公司都将演变为 AaaS（Agentic as a Service）——即以智能体为核心的服务平台。
不过，智能体系统进入企业网络也带来了新的安全挑战。因为这些系统不仅能够访问敏感数据，还可以执行代码并与外部网络通信。如果缺乏安全机制，可能带来严重风险。为此，NVIDIA 与 OpenClaw 作者 Peter Steinberger 以及多位安全与计算专家合作，对系统进行了企业级安全扩展，并推出 NVIDIA NemoClaw 参考架构。
图片
该架构在 OpenClaw 基础上加入了名为 OpenShell 的安全组件，并提供企业级策略执行、网络防护和隐私路由等能力，使企业能够安全地部署和运行智能体系统。同时，系统还支持连接企业已有的策略引擎和治理工具，从而在确保合规和数据安全的前提下运行 AI 智能体。
9
 推进开放模型生态
此外，NVIDIA 还在推进开放模型生态。黄仁勋表示，现实世界的需求高度多样化，不可能由单一模型满足所有行业。因此，开放模型正在形成一个规模庞大的 AI 生态系统，目前已经包含 接近 300 万个开放模型，覆盖语言、视觉、生物、物理和自动驾驶等多个领域。作为其中的重要贡献者，NVIDIA 已发布多条开放模型产品线，包括 Nemotron（语言模型）、Cosmos World Foundation Model（世界模型）、Project GR00T（机器人基础模型）、Drive AV Foundation Models（自动驾驶模型）、BioNeMo（数字生物学模型）以及 Earth-2（AI 物理与气候模拟平台），并同时开放训练数据、训练方法和框架工具，以推动整个 AI 生态的发展。
黄仁勋表示，NVIDIA 的开放模型之所以能够在多个榜单中处于领先位置，不仅因为其性能达到世界级水平，更重要的是公司会持续投入长期研发。“我们不会停止改进这些模型，”他说。例如 Nemotron 模型已经从 Nemotron 3 走向 Nemotron 4，Cosmos World Foundation Model 也从 Cosmos 1 发展到 Cosmos 2，而机器人模型 Project GR00T 也在不断迭代。NVIDIA 的策略是“纵向整合、横向开放”，在持续提升模型能力的同时，让整个生态都能参与到 AI 发展中来。
他还展示了 Nemotron 3 在智能体框架 OpenClaw 中的表现。根据公开评测数据，当前全球排名前三的模型均处于这一技术前沿。黄仁勋表示，NVIDIA 不仅希望构建领先的基础模型，更重要的是让开发者能够在此基础上进行微调和后训练，构建适用于不同领域的专用 AI 系统。为此，NVIDIA 推出了 Nemotron 3 Ultra 作为新一代基础模型，并希望借此帮助各个国家和行业构建属于自己的 “主权 AI（Sovereign AI）”。
为了进一步推动这一生态，NVIDIA 在大会上宣布成立 Nemotron Coalition。该联盟将与多家技术公司合作，共同推进 Nemotron 系列模型的发展。参与合作的公司包括图像技术公司 Black Forest Labs、AI 编程平台 Cursor、智能体开发框架 LangChain、欧洲 AI 公司 Mistral AI、AI 搜索平台 Perplexity AI、印度 AI 公司 Sarvam AI 以及 Thinking Machines Lab 等。黄仁勋表示，随着越来越多企业加入合作，AI 模型将能够覆盖从语言到生物、从物理到自动驾驶等广泛领域。
在企业软件层面，黄仁勋再次强调，未来所有公司都需要制定自己的 OpenClaw 战略。随着智能体系统的发展，传统的 SaaS 软件模式将逐渐转向 Agentic as a Service（AaaS）。企业不仅会使用 token 来增强员工生产力，还会通过 AI 工厂生产 token，并向客户提供智能体服务。他甚至预测，未来科技公司招聘工程师时，除了薪资外，还会提供“token 配额”，因为拥有更多 AI 计算资源的工程师能够获得更高的生产效率。
除了数字智能体，NVIDIA 还在推进 物理 AI（Physical AI）。黄仁勋表示，目前全球几乎所有机器人公司都在与 NVIDIA 合作，现场展示的机器人数量超过 100 台。NVIDIA 为机器人开发提供完整技术体系，包括训练计算平台、合成数据与仿真平台，以及部署在机器人内部的计算系统。同时，公司还提供完整的软件和模型生态，例如机器人仿真与训练平台 NVIDIA Isaac Lab、世界模型 Cosmos World Foundation Model 以及机器人基础模型 Project GR00T。
图片
在自动驾驶领域，黄仁勋表示“自动驾驶的 ChatGPT 时刻已经到来”。基于 NVIDIA Drive AV 和相关模型体系，车辆现在已经具备推理能力，可以解释自己的驾驶决策并执行语音指令。NVIDIA 还宣布新的 Robotaxi 合作伙伴，包括 BYD、Hyundai Motor Company、Nissan 和 Geely，这些公司每年合计生产约 1800 万辆汽车。同时，NVIDIA 还将与 Uber 合作，在多个城市部署自动驾驶出租车网络。
在机器人产业方面，NVIDIA 正与 ABB、Universal Robots、KUKA 等企业合作，将物理 AI 模型与仿真系统结合，用于工业生产线自动化。黄仁勋还提到，未来通信基础设施也将成为 AI 系统的一部分，例如 T-Mobile 的通信塔未来可能演变为“机器人 AI 基站”，能够实时分析交通和网络情况并动态调整信号。
在总结演讲时，黄仁勋表示，AI 产业正同时经历三大变革：AI 推理与 AI 工厂、智能体系统革命，以及物理 AI 与机器人时代。随着这些技术逐渐成熟，计算能力、AI 模型和基础设施将共同推动全球产业进入新的发展阶段
"""
with open("secret_doc.txt", "w", encoding="utf-8") as f:
    f.write(secret_text)

# 2. 撕碎并建立向量数据库
loader = TextLoader("secret_doc.txt", encoding="utf-8")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

# 存入 Chroma 数据库
vector_db = Chroma.from_documents(chunks, embedding_radar)
# 注意：把搜索范围 k 调大到 4，确保不会漏掉情报！
retriever = vector_db.as_retriever(search_kwargs={"k": 4}) 

print("✅ 机密资料已向量化并锁入数据库！检索雷达已上线！")

In [ ]:
# 👉 在这里输入您的问题：
user_question = "黄仁勋讲了什么，总结一下。同时你觉得2026年的AI硬件市场，除了英伟达，谁还能打？"

# 1. 雷达瞬间检索
retrieved_docs = retriever.invoke(user_question)

# 2. 组装提示词
prompt_template = """你是一位资深的科技战略分析师。
请结合提供的<背景参考资料>，用专业的口吻回答指挥官的问题。

【指令规范】：
1. 优先使用资料中的数据（如性能百分比、具体代号）。
2. 在资料基础上，你可以进行合理的逻辑总结和技术关联。
3. 如果资料中完全没有提及某个关键点，请明确指出。

<背景参考资料>
{context}
</机密情报>

指挥官的问题：{question}
深度战术分析："""
prompt = PromptTemplate.from_template(prompt_template)
context_text = "\n".join([doc.page_content for doc in retrieved_docs])
final_prompt = prompt.format(context=context_text, question=user_question)

# 3. 千问副官开火
print(f"收到指令：【{user_question}】\n")
response = llm.invoke(final_prompt)
print("千问 副官回答：")
print(response)